# マスク言語モデルで文の自然さを測る (疑似対数尤度)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_transformers/mlm_scoring.ipynb)

NLP クックブック 11-14 ／ Transformers の基本操作

BERT のような双方向モデルでも文の自然さを数値化したい。1 トークンずつマスクして、元のトークンの確率を合計する。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

name = "tohoku-nlp/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForMaskedLM.from_pretrained(name).eval()

@torch.no_grad()
def pll(text):
    """疑似対数尤度 (pseudo-log-likelihood) と、トークンごとの内訳"""
    ids = tokenizer(text, return_tensors="pt")["input_ids"][0]
    n = len(ids) - 2                                        # [CLS] と [SEP] を除く
    batch = ids.repeat(n, 1)
    pos = torch.arange(1, n + 1)
    batch[torch.arange(n), pos] = tokenizer.mask_token_id   # i 番目の文は i 番目のトークンだけマスク
    logp = model(batch).logits[torch.arange(n), pos].log_softmax(-1)
    scores = logp[torch.arange(n), ids[pos]]
    return scores.sum().item(), [(t, round(v, 1)) for t, v in zip(tokenizer.convert_ids_to_tokens(ids[pos]), scores.tolist())]

for text in ["太郎は図書館で本を読んだ。", "太郎は図書館が本を読んだ。", "太郎は図書館で本を飲んだ。"]:
    total, detail = pll(text)
    print(f"{total:7.2f}  {text}")
    print("        ", detail)

 -11.13  太郎は図書館で本を読んだ。
         [('太郎', -9.2), ('は', -0.3), ('図書', -0.1), ('館', -0.7), ('で', -0.1), ('本', -0.6), ('を', -0.0), ('読ん', -0.1), ('だ', -0.0), ('。', -0.0)]
 -25.22  太郎は図書館が本を読んだ。
         [('太郎', -9.6), ('は', -1.2), ('図書', -0.8), ('館', -1.5), ('が', -10.9), ('本', -0.5), ('を', -0.1), ('読ん', -0.5), ('だ', -0.0), ('。', -0.0)]
 -24.69  太郎は図書館で本を飲んだ。
         [('太郎', -9.6), ('は', -0.2), ('図書', -0.2), ('館', -0.8), ('で', -0.2), ('本', -4.4), ('を', -0.0), ('飲ん', -9.3), ('だ', -0.0), ('。', -0.0)]


- 自己回帰モデルの対数尤度と違い、左右両方の文脈を使って各トークンを評価する。誤りのある位置のスコアが局所的に下がるので、誤り検出に使いやすい。
- 文長分の推論が必要 (文長 n のバッチ 1 回)。長い文や大量の文には重い。
- 確率として正規化されていない「疑似」尤度なので、あくまで相対比較に使う (Salazar et al., 2020)。